# 04 - Análisis con SQL

**Proyecto:** Analítica de ventas de videojuegos en Databricks  
**Autor:** Darren J. Blackman M.  
**Asignatura:** Análisis de Datos y Toma de Decisiones en Computación

## Objetivo
Demostrar análisis mediante Spark SQL, incluyendo agrupaciones, subconsultas y funciones de ventana.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

CATALOGO = spark.sql("SELECT current_catalog()").first()[0]
ESQUEMA = "default"

def tabla(nombre):
    return f"`{CATALOGO}`.`{ESQUEMA}`.`{nombre}`"

print(f"Catálogo activo: {CATALOGO}")
print(f"Esquema utilizado: {ESQUEMA}")

Catálogo activo: workspace
Esquema utilizado: default


In [0]:
df = spark.table(tabla("vgsales_limpio"))
df.createOrReplaceTempView("vgsales_limpio_sql")
print("Vista temporal creada: vgsales_limpio_sql")

Vista temporal creada: vgsales_limpio_sql


In [0]:
%sql
SELECT
  COUNT(*) AS total_videojuegos,
  ROUND(SUM(Global_Sales), 2) AS ventas_globales,
  MIN(Year) AS anio_minimo,
  MAX(Year) AS anio_maximo
FROM vgsales_limpio_sql;

total_videojuegos,ventas_globales,anio_minimo,anio_maximo
16327,8820.36,1980,2020


In [0]:
%sql
SELECT
  Platform,
  COUNT(*) AS cantidad_juegos,
  ROUND(SUM(Global_Sales), 2) AS ventas_globales,
  ROUND(AVG(Global_Sales), 3) AS promedio_por_juego
FROM vgsales_limpio_sql
GROUP BY Platform
ORDER BY ventas_globales DESC
LIMIT 10;

Platform,cantidad_juegos,ventas_globales,promedio_por_juego
PS2,2127,1233.46,0.58
X360,1235,969.61,0.785
PS3,1304,949.35,0.728
Wii,1290,909.81,0.705
DS,2133,818.96,0.384
PS,1189,727.39,0.612
GBA,811,313.56,0.387
PSP,1197,291.71,0.244
PS4,336,278.1,0.828
PC,943,255.05,0.27


In [0]:
%sql
SELECT
  Genre,
  COUNT(*) AS cantidad_juegos,
  ROUND(SUM(Global_Sales), 2) AS ventas_globales
FROM vgsales_limpio_sql
GROUP BY Genre
ORDER BY ventas_globales DESC;

Genre,cantidad_juegos,ventas_globales
Action,3253,1722.88
Sports,2304,1309.24
Shooter,1282,1026.2
Role-Playing,1471,923.84
Platform,876,829.15
Misc,1710,797.62
Racing,1226,726.77
Fighting,836,444.05
Simulation,851,390.16
Puzzle,571,242.22


In [0]:
%sql
SELECT
  Year,
  ROUND(SUM(Global_Sales), 2) AS ventas_globales
FROM vgsales_limpio_sql
GROUP BY Year
ORDER BY Year;

Year,ventas_globales
1980,11.38
1981,35.77
1982,28.86
1983,16.79
1984,50.36
1985,53.94
1986,37.07
1987,21.74
1988,47.22
1989,73.45


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT Region, ROUND(SUM(Ventas), 2) AS Ventas
FROM (
  SELECT stack(
    4,
    'Norteamérica', NA_Sales,
    'Europa', EU_Sales,
    'Japón', JP_Sales,
    'Otras regiones', Other_Sales
  ) AS (Region, Ventas)
  FROM vgsales_limpio_sql
)
GROUP BY Region
ORDER BY Ventas DESC;

Region,Ventas
Norteamérica,4333.43
Europa,2409.12
Japón,1284.3
Otras regiones,789.01


In [0]:
%sql
WITH ventas_plataforma AS (
  SELECT Platform, SUM(Global_Sales) AS ventas
  FROM vgsales_limpio_sql
  GROUP BY Platform
)
SELECT
  Platform,
  ROUND(ventas, 2) AS ventas,
  DENSE_RANK() OVER (ORDER BY ventas DESC) AS posicion
FROM ventas_plataforma
ORDER BY posicion;

Platform,ventas,posicion
PS2,1233.46,1
X360,969.61,2
PS3,949.35,3
Wii,909.81,4
DS,818.96,5
PS,727.39,6
GBA,313.56,7
PSP,291.71,8
PS4,278.1,9
PC,255.05,10


In [0]:
%sql
SELECT Name, Platform, Genre, Year, Global_Sales
FROM vgsales_limpio_sql
WHERE Global_Sales > (
  SELECT AVG(Global_Sales) + 2 * STDDEV(Global_Sales)
  FROM vgsales_limpio_sql
)
ORDER BY Global_Sales DESC
LIMIT 20;

Name,Platform,Genre,Year,Global_Sales
Wii Sports,Wii,Sports,2006,82.74
Super Mario Bros.,NES,Platform,1985,40.24
Mario Kart Wii,Wii,Racing,2008,35.82
Wii Sports Resort,Wii,Sports,2009,33.0
Pokemon Red/Pokemon Blue,GB,Role-Playing,1996,31.37
Tetris,GB,Puzzle,1989,30.26
New Super Mario Bros.,DS,Platform,2006,30.01
Wii Play,Wii,Misc,2006,29.02
New Super Mario Bros. Wii,Wii,Platform,2009,28.62
Duck Hunt,NES,Shooter,1984,28.31


In [0]:
%sql
WITH ventas_region AS (
  SELECT Genre,
         SUM(NA_Sales) AS NA,
         SUM(EU_Sales) AS EU,
         SUM(JP_Sales) AS JP,
         SUM(Other_Sales) AS Otras
  FROM vgsales_limpio_sql
  GROUP BY Genre
)
SELECT *
FROM ventas_region
ORDER BY NA + EU + JP + Otras DESC;

Genre,NA,EU,JP,Otras
Action,861.7999999999975,516.4799999999949,158.66000000000054,184.92000000000033
Sports,670.0899999999987,371.33999999999884,134.76000000000025,132.65000000000083
Shooter,575.1599999999985,310.4499999999996,38.18000000000003,101.90000000000026
Role-Playing,326.4999999999998,187.58000000000044,350.2899999999998,59.380000000000074
Platform,445.9899999999995,200.67000000000024,130.65000000000015,51.50999999999995
Misc,402.4800000000003,213.82000000000042,106.66999999999999,74.02000000000004
Racing,356.93,236.32000000000028,56.61000000000004,76.68000000000016
Fighting,220.73999999999998,100.00000000000013,87.14999999999998,36.190000000000076
Simulation,181.78000000000003,113.20000000000007,63.54000000000003,31.360000000000078
Puzzle,122.01000000000009,50.53000000000005,56.679999999999964,12.469999999999951
